In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

from mcfs.load_runs import load_saved_fake_spectra

In [ ]:
data = load_saved_fake_spectra(
    hdf5_path="/home/STORAGE/SKEWERS/TNG50-4/snapdir_033/grid_TNG50-4_snap033_nspec2_axis2_nbins1024_test.hdf5",
    base="/home/STORAGE/TNG50-4",
    num=33,
    axis=2,
    use_res=None,
)

In [ ]:
print("\n=== Loaded paths ===")
for k, v in data["paths"].items():
    print(f"{k:10s}: {v}")

print("\n=== Loaded lines ===")
print(data["lines"])

print("\n=== fake_spectra metadata ===")
for k, v in data["meta"].items():
    print(f"{k:10s}: {v}")

print("\n=== Timing info ===")
for k, v in data["timing"].items():
    print(f"{k:18s}: {v}")

print("\n=== Array keys ===")
print("tau :", list(data["tau"].keys()))
print("n   :", list(data["n"].keys()))
print("T   :", list(data["T"].keys()))
print("vlos:", list(data["vlos"].keys()))

In [ ]:
sp = data["sp"]
tau = data["tau"]
n = data["n"]
T = data["T"]
vlos = data["vlos"]
lines = data["lines"]

In [ ]:
import mcfs

import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = mcfs.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
# %matplotlib widget

In [ ]:
ii_skewer = 0  # choose skewer index

# Build keys in the same order as lines list
keys = [f"{e}{i}_{lam}" for (e, i, lam) in lines]

# x-axis in velocity units
v_kms = np.arange(sp.nbins) * sp.dvbin
x = v_kms
xlabel = r"Distance along LOS [km/s]"

# Colors per key
cmap = plt.get_cmap("tab10")
colors = {k: cmap(i % cmap.N) for i, k in enumerate(keys)}

fig, axes = plt.subplots(5, 1, figsize=(11, 13), sharex=True)

# --- panels: n, T, vlos, tau, flux ---
for k in keys:
    if k in n:
        axes[0].plot(x, n[k][ii_skewer], color=colors[k], lw=1.3)
    if k in T:
        axes[1].plot(x, T[k][ii_skewer], color=colors[k], lw=1.3)
    if k in vlos:
        axes[2].plot(x, vlos[k][ii_skewer], color=colors[k], lw=1.3)
    if k in tau:
        axes[3].plot(x, tau[k][ii_skewer], color=colors[k], lw=1.3)
        axes[4].plot(x, np.exp(-tau[k][ii_skewer]), color=colors[k], lw=1.3)

axes[0].set_ylabel(r"$n$ [cm$^{-3}$]")
axes[0].set_yscale("log")

axes[1].set_ylabel(r"$T$ [K]")
axes[1].set_yscale("log")

axes[2].set_ylabel(r"$v_{\parallel}$ [km/s]")

axes[3].set_ylabel(r"$\tau$")
axes[3].set_yscale("log")

axes[4].set_ylabel(r"$F/F_0$")
axes[4].set_xlabel(xlabel)

handles = [Line2D([0], [0], color=colors[k], lw=3, label=k) for k in keys]
axes[0].legend(handles=handles, loc="upper right", frameon=True, fontsize=12, title="Line key")

plt.tight_layout()
plt.show()

In [ ]:
keys = [f"{e}{i}_{lam}" for (e, i, lam) in lines]

# --- choose which skewers to plot ---
mode = "random"     # "random", "first", or "given"
Nplot = 20
seed = 137
axis_select = None  # 1, 2, 3, or None
ii_list = [0, 1, 2, 3]   # only used if mode == "given"

# --- style ---
alpha = 0.6
lw = 1.0
ylog_n = True
ylog_T = True
ylog_tau = True

# --- allowed indices ---
all_idx = np.arange(sp.NumLos)

if axis_select is not None:
    if not hasattr(sp, "axis"):
        raise AttributeError("sp has no attribute 'axis' (needed for axis_select).")
    allowed = np.where(sp.axis == axis_select)[0]
    if allowed.size == 0:
        raise ValueError(f"No skewers found with axis_select={axis_select}.")
else:
    allowed = all_idx

# --- choose ii_list ---
if mode == "random":
    rng = np.random.default_rng(seed)
    ii_list = rng.choice(allowed, size=min(Nplot, allowed.size), replace=False)
elif mode == "first":
    ii_list = allowed[: min(Nplot, allowed.size)]
elif mode == "given":
    ii_list = np.array(ii_list, dtype=int)
    bad = np.setdiff1d(ii_list, allowed)
    if bad.size > 0:
        raise ValueError(f"Some indices are not allowed: {bad[:10]}")
else:
    raise ValueError("mode must be one of: 'random', 'first', 'given'.")

print(f"Plotting {len(ii_list)} skewers | mode={mode} | axis_select={axis_select}")
print("Indices:", ii_list[:30], "..." if len(ii_list) > 30 else "")

# --- x-axis in velocity units (km/s) ---
x = np.arange(sp.nbins) * sp.dvbin
xlabel = r"Distance along LOS [$\mathrm{km\,s^{-1}}$]"

# --- print mean flux across ALL skewers/pixels ---
print(f"\nGlobal means over all skewers and pixels (NumLos={sp.NumLos}, nbins={sp.nbins}):")
for k in keys:
    if k in tau:
        F = np.exp(-tau[k])
        print(f"  {k:10s}  <F> = {float(F.mean()):.6f}")

# --- plotting ---
cmap = plt.get_cmap("tab10")
colors = {k: cmap(i % cmap.N) for i, k in enumerate(keys)}

fig, axes = plt.subplots(5, 1, figsize=(12, 14), sharex=True)

# Panel 1: density
for k in keys:
    if k not in n:
        print(f"[warning] missing density for {k}")
        continue
    for ii in ii_list:
        axes[0].plot(x, n[k][ii], color=colors[k], lw=lw, alpha=alpha)
axes[0].set_ylabel(r"$n$ [cm$^{-3}$]")
if ylog_n:
    axes[0].set_yscale("log")

# Panel 2: temperature
for k in keys:
    if k not in T:
        print(f"[warning] missing temperature for {k}")
        continue
    for ii in ii_list:
        axes[1].plot(x, T[k][ii], color=colors[k], lw=lw, alpha=alpha)
axes[1].set_ylabel(r"$T$ [K]")
if ylog_T:
    axes[1].set_yscale("log")

# Panel 3: LOS velocity
for k in keys:
    if k not in vlos:
        print(f"[warning] missing LOS velocity for {k}")
        continue
    for ii in ii_list:
        axes[2].plot(x, vlos[k][ii], color=colors[k], lw=lw, alpha=alpha)
axes[2].set_ylabel(r"$v_{\parallel}$ [km/s]")

# Panel 4: tau
for k in keys:
    if k not in tau:
        print(f"[warning] missing tau for {k}")
        continue
    for ii in ii_list:
        axes[3].plot(x, tau[k][ii], color=colors[k], lw=lw, alpha=alpha)
axes[3].set_ylabel(r"$\tau$")
if ylog_tau:
    axes[3].set_yscale("log")
axes[3].set_title(f"{len(ii_list)} skewers | z={sp.red:.3f}")

# Panel 5: flux
for k in keys:
    if k not in tau:
        continue
    for ii in ii_list:
        axes[4].plot(x, np.exp(-tau[k][ii]), color=colors[k], lw=lw, alpha=alpha)
axes[4].set_ylabel(r"$e^{-\tau}$")
axes[4].set_xlabel(xlabel)

# Legend: one entry per line
handles = [Line2D([0], [0], color=colors[k], lw=3, label=k) for k in keys]
axes[0].legend(handles=handles, loc="upper right", frameon=True, fontsize=12, title="Line key")

plt.tight_layout()
plt.show()